In [1]:
!git clone https://github.com/timofeykhodykin/ai360-financial-qa.git

Cloning into 'ai360-financial-qa'...
remote: Enumerating objects: 879, done.
remote: Counting objects: 100% (46/46), done.
remote: Compressing objects: 100% (31/31), done.
remote: Total 879 (delta 23), reused 26 (delta 15), pack-reused 833 (from 1)
Receiving objects: 100% (879/879), 313.87 MiB | 38.27 MiB/s, done.
Resolving deltas: 100% (348/348), done.
Updating files: 100% (277/277), done.


In [2]:
import logging
from transformers import logging as tf_logging

tf_logging.set_verbosity_error()

In [3]:
import sys

In [4]:
sys.path.append('/kaggle/working/ai360-financial-qa/')

In [5]:
import asyncio
import json
import os
import time
from pathlib import Path

import dotenv
from tqdm import tqdm

from financial_qa.chunkers import SlidingWindowChunker
from financial_qa.embedders import HFEmbedder
from financial_qa.rag import RAG
from financial_qa.agent.agent_loop import OpenRouterAgentLoop
from financial_qa.evaluation import evaluate_async, load_jsonl

In [6]:
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
OPENROUTER_API_KEY = user_secrets.get_secret("OPENROUTER_API_KEY")

os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY

In [7]:
dotenv.load_dotenv('.env')
OPENROUTER_API_KEY = os.getenv('OPENROUTER_API_KEY')
if not OPENROUTER_API_KEY:
    raise ValueError('OPENROUTER_API_KEY is required')

DATASET_FILE = '/kaggle/working/ai360-financial-qa/dataset.jsonl'
DATASET_SPLIT = None
MAX_QUESTIONS = None

RAG_DB = 'default'
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200
TOP_K = 5
EMBED_MODEL = 'intfloat/multilingual-e5-small'

GEN_MODEL = 'google/gemini-2.0-flash-lite-001'
QUERY_CONCURRENCY = 4

JUDGE_MODEL = 'google/gemini-2.0-flash-lite-001'
JUDGE_PROCESSES = None  # None => one process per question

In [8]:
all_records = load_jsonl(DATASET_FILE)

In [9]:
len(all_records)

449

In [10]:
all_records = load_jsonl(DATASET_FILE)

all_records = [
    all_records[r]
    for r in all_records 
    if DATASET_SPLIT is None or all_records[r].get('split') == DATASET_SPLIT
]

seen = set()
records = []
cnt = 0
for r in all_records:
    if r['question_id'] not in seen:
        records.append(r)
        seen.add(r['question_id'])
        cnt += 1
    else:
        print('huy')
print(cnt)

if MAX_QUESTIONS:
    records = records[:MAX_QUESTIONS]

golden = {r['question_id']: r for r in records}
print(f'Loaded {len(records)} records (split={DATASET_SPLIT!r})')
print('Sample:', json.dumps(records[0], ensure_ascii=False, indent=2))


449
Loaded 449 records (split=None)
Sample: {
  "question_id": "q_00d660efcf3e4607",
  "question": "Какова общая сумма финансовых обязательств ЗАО «Альфа-Банк», подлежащих переводу на альтернативные процентные базовые ставки, по состоянию на 31 декабря 2025 года?",
  "split": "test",
  "gold_evidence": [
    {
      "doc_id": "alfa_2025_annual",
      "pages": [
        103
      ]
    }
  ],
  "gold_answer": "1,151 тыс. белорусских рублей"
}


In [11]:
chunker = SlidingWindowChunker(chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP)
embedder = HFEmbedder(model_name=EMBED_MODEL)
rag = RAG(
    chunker=chunker,
    embedder=embedder,
    data_dir='data/parsed',
    store_dir='indexes',
    name=RAG_DB,
    top_k=TOP_K,
)

store_dir = Path('/kaggle/working/ai360-financial-qa/indexes') / RAG_DB
has_index = store_dir.exists() and any(store_dir.glob('*.npz'))
if not has_index:
    print('No index found; running precalc...')
    rag.precalc()
else:
    print(f'Using existing index at {store_dir}')


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/471M [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Using existing index at /kaggle/working/ai360-financial-qa/indexes/default


In [12]:
loop = OpenRouterAgentLoop(rag=rag, model=GEN_MODEL)
print(f'Agent log file: {loop.log_root}')

Agent log file: logs/agent


In [13]:
async def run_queries(records):
    predictions = {}
    errors = []
    timings = []
    semaphore = asyncio.Semaphore(QUERY_CONCURRENCY)

    async def _query_one(rec):
        start = time.perf_counter()
        try:
            answer, confidence = await loop.aquery(rec['question'], rec['question_id'])
            error = None
        except Exception as e:
            answer = ''
            confidence = None
            error = str(e)
        elapsed = time.perf_counter() - start
        return {
            'question_id': rec['question_id'],
            'question': rec['question'],
            'answer': answer,
            'evidence': [],
            'confidence': confidence,
            'error': error,
            'elapsed_s': elapsed,
        }

    async def _bound(rec):
        async with semaphore:
            return await _query_one(rec)

    tasks = {asyncio.create_task(_bound(rec)): rec for rec in records}
    progress = tqdm(total=len(tasks), desc='Querying agent', unit='question')
    for task in asyncio.as_completed(tasks):
        result = await task
        predictions[result['question_id']] = result
        if result['error']:
            errors.append(result)
        timings.append(result['elapsed_s'])
        progress.update(1)
        progress.set_postfix(
            errors=len(errors),
            avg_s=f"{sum(timings)/len(timings):.2f}",
            last_conf=result['confidence'],
        )
    progress.close()
    return predictions, errors

predicted, query_errors = await run_queries(records)
print(f'Done: {len(predicted)} answers, {len(query_errors)} errors')


Querying agent: 100%|██████████| 449/449 [03:14<00:00,  2.31question/s, avg_s=1.73, errors=0, last_conf=0.889]

Done: 449 answers, 0 errors


In [14]:
result = await evaluate_async(
    golden=golden,
    predicted=predicted,
    model=JUDGE_MODEL,
    api_key=OPENROUTER_API_KEY,
    detailed_result=True,
    include_evidence=False,
    use_processes=True,
    max_workers=JUDGE_PROCESSES,
    progress_desc='LLM-as-judge',
)

LLM-as-judge: 100%|██████████| 449/449 [00:06<00:00, 69.83question/s, accuracy=38.53%, correct=173, errors=0]


In [15]:
result['correct'] / result['total']

0.38530066815144765

In [16]:
for res in result['results'][0:3]:
    print(res)
    print('-' * 75)

{'question_id': 'q_009c97884b8dc010', 'question': 'Каковы чистые комиссионные доходы Банка ДомРФ за шесть месяцев, закончившихся 30 июня 2025 года?', 'gold_answer': '5 592 млн руб.', 'predicted_answer': 'Чистые комиссионные доходы Банка ДомРФ за шесть месяцев, закончившихся 30 июня 2025 года, составили 5 592 млн.\n', 'judge_score': 1, 'judge_reasoning': 'Предсказанный ответ содержит ту же информацию, что и золотой ответ, с незначительными изменениями в формулировке.'}
---------------------------------------------------------------------------
{'question_id': 'q_00d660efcf3e4607', 'question': 'Какова общая сумма финансовых обязательств ЗАО «Альфа-Банк», подлежащих переводу на альтернативные процентные базовые ставки, по состоянию на 31 декабря 2025 года?', 'gold_answer': '1,151 тыс. белорусских рублей', 'predicted_answer': 'Я не могу ответить на этот вопрос, так как в предоставленных фрагментах нет информации об общей сумме финансовых обязательств ЗАО «Альфа-Банк», подлежащих переводу н

In [17]:
import shutil
shutil.make_archive('logs', 'zip', '/kaggle/working/logs')

'/kaggle/working/logs.zip'